# Cross-channel PatchTST

A controlled PatchTST ablation that adds cross-channel attention after each unchanged temporal transformer block.

In [ ]:
#| default_exp cross_channel_patchtst

In [ ]:
#| export
import math

import torch
from torch import Tensor, nn

from physiojepa.augmentations import unpatch
from physiojepa.heads import AttentiveClassifier
from physiojepa.layers import MultiHeadAttention
from physiojepa.patchtst import PatchTFTSimple

In [ ]:
#| export
class CrossChannelAttentionBlock(nn.Module):
    """Residual self-attention across channels at one time patch."""

    def __init__(
        self,
        d_model,
        n_heads,
        attn_dropout=0.0,
        dropout=0.0,
        bias=True,
        pre_norm=False,
    ):
        super().__init__()
        assert not d_model % n_heads, (
            f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        )
        self.self_attn = MultiHeadAttention(
            dim=d_model,
            num_heads=n_heads,
            qkv_bias=bias,
            qk_scale=None,
            attn_drop=attn_dropout,
            proj_drop=dropout,
            rotary_pes=False,
        )
        self.dropout_attn = nn.Dropout(dropout)
        self.norm_attn = nn.LayerNorm(d_model)
        self.pre_norm = pre_norm

    def forward(self, src: Tensor):
        if self.pre_norm:
            src = self.norm_attn(src)
        src = src + self.dropout_attn(self.self_attn(src, mask=None))
        if not self.pre_norm:
            src = self.norm_attn(src)
        return src

In [ ]:
#| export
class PatchTFTCrossChannel(PatchTFTSimple):
    """PatchTST with cross-channel attention after every temporal block."""

    def __init__(
        self,
        c_in,
        patch_size,
        patch_stride,
        num_patches,
        d_model,
        n_heads,
        d_ff,
        num_layers,
        augmentations=None,
        mask_ratio=0.1,
        shared_embedding=False,
        pretrain_head=True,
        dropout=0.0,
        attn_dropout=0.0,
        act='gelu',
        pre_norm=False,
        pe_type='tAPE',
        qkv_bias=True,
        init_std=0.02,
        tokenizer_type='simple',
        tokenizer_kwargs=None,
    ):
        if augmentations is None:
            augmentations = []
        if tokenizer_kwargs is None:
            tokenizer_kwargs = {}
        super().__init__(
            c_in=c_in,
            patch_size=patch_size,
            patch_stride=patch_stride,
            num_patches=num_patches,
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            augmentations=augmentations,
            mask_ratio=mask_ratio,
            shared_embedding=shared_embedding,
            pretrain_head=pretrain_head,
            dropout=dropout,
            attn_dropout=attn_dropout,
            act=act,
            pre_norm=pre_norm,
            pe_type=pe_type,
            qkv_bias=qkv_bias,
            init_std=init_std,
            tokenizer_type=tokenizer_type,
            tokenizer_kwargs=tokenizer_kwargs,
        )
        self.channel_layers = nn.ModuleList([
            CrossChannelAttentionBlock(
                d_model=d_model,
                n_heads=n_heads,
                attn_dropout=attn_dropout,
                dropout=dropout,
                bias=qkv_bias,
                pre_norm=pre_norm,
            )
            for _ in range(num_layers)
        ])
        self.channel_layers.apply(self._init_weights)
        self._rescale_channel_blocks()

    def _rescale_channel_blocks(self):
        for layer_id, layer in enumerate(self.channel_layers, start=1):
            layer.self_attn.proj.weight.data.div_(math.sqrt(2.0 * layer_id))

    def forward(self, x):
        """Encode ``[batch, channels, samples]`` with temporal/channel blocks."""
        bs = x.size(0)
        seq_len = x.size(-1)
        x = self.patch_layer(x, constant_pad=True, constant_pad_value=0)
        y_true = x.clone().detach()

        if self.training and self.pretrain_head:
            z = self.mask(x)
        else:
            z = x
        if self.tokenizer_type != 'linear':
            z = unpatch(z, seq_len, remove_padding=True)
        z = self.tokenizer(z)
        if z.dim() == 3:
            z = z.unsqueeze(2)

        z = z.transpose(1, 2)
        transformer_c_in = z.size(1)
        z = z.reshape(bs * transformer_c_in, self.num_patches, self.d_model)
        z = self.dropout(self.pe(z))
        z = z.reshape(bs, transformer_c_in, self.num_patches, self.d_model)

        for temporal_layer, channel_layer in zip(self.layers, self.channel_layers):
            temporal_z = z.reshape(
                bs * transformer_c_in, self.num_patches, self.d_model
            )
            temporal_z = temporal_layer(temporal_z, mask=None)
            z = temporal_z.reshape(
                bs, transformer_c_in, self.num_patches, self.d_model
            )

            channel_z = z.permute(0, 2, 1, 3).contiguous().reshape(
                bs * self.num_patches, transformer_c_in, self.d_model
            )
            channel_z = channel_layer(channel_z)
            z = channel_z.reshape(
                bs, self.num_patches, transformer_c_in, self.d_model
            ).permute(0, 2, 1, 3).contiguous()

        z = z.permute(0, 1, 3, 2)
        if self.pretrain_head:
            y_pred = self.head(z)
            return z, y_pred, y_true
        return z

In [ ]:
#| export
class SupervisedPatchTSTCrossChannel(nn.Module):
    """End-to-end supervised PatchTST with cross-channel attention."""

    def __init__(
        self,
        c_in,
        patch_size,
        patch_stride,
        num_patches,
        d_model,
        n_heads,
        d_ff,
        num_layers,
        augmentations=None,
        mask_ratio=0.0,
        shared_embedding=False,
        dropout=0.0,
        attn_dropout=0.0,
        act='gelu',
        pre_norm=False,
        pe_type='tAPE',
        qkv_bias=True,
        init_std=0.02,
        tokenizer_type='simple',
        tokenizer_kwargs=None,
        classifier_mlp_ratio=4.0,
        classifier_depth=1,
        classifier_init_std=0.02,
        classifier_qkv_bias=True,
        classifier_complete_block=True,
        classifier_affine=False,
        num_classes=1,
    ):
        super().__init__()
        if augmentations is None:
            augmentations = []
        if tokenizer_kwargs is None:
            tokenizer_kwargs = {}
        self.encoder = PatchTFTCrossChannel(
            c_in=c_in,
            patch_size=patch_size,
            patch_stride=patch_stride,
            num_patches=num_patches,
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            augmentations=augmentations,
            mask_ratio=mask_ratio,
            shared_embedding=shared_embedding,
            pretrain_head=False,
            dropout=dropout,
            attn_dropout=attn_dropout,
            act=act,
            pre_norm=pre_norm,
            pe_type=pe_type,
            qkv_bias=qkv_bias,
            init_std=init_std,
            tokenizer_type=tokenizer_type,
            tokenizer_kwargs=tokenizer_kwargs,
        )
        self.classifier = AttentiveClassifier(
            embed_dim=d_model,
            num_heads=n_heads,
            mlp_ratio=classifier_mlp_ratio,
            depth=classifier_depth,
            init_std=classifier_init_std,
            qkv_bias=classifier_qkv_bias,
            num_classes=num_classes,
            complete_block=classifier_complete_block,
            affine=classifier_affine,
            c_in=c_in,
        )

    def forward(self, x):
        return self.classifier(self.encoder(x))

In [ ]:
# Focused regression checks: parity when bypassed, channel mixing, and gradients.
torch.manual_seed(16)
encoder_kwargs = dict(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=2,
    shared_embedding=False,
    pretrain_head=False,
    dropout=0.0,
    attn_dropout=0.0,
    pe_type='rotary',
    tokenizer_type='linear',
)
base = PatchTFTSimple(**encoder_kwargs).eval()
cross = PatchTFTCrossChannel(**encoder_kwargs).eval()
base_state = base.state_dict()
cross_base_state = {
    key: value
    for key, value in cross.state_dict().items()
    if not key.startswith('channel_layers.')
}
assert base_state.keys() == cross_base_state.keys()
assert all(base_state[key].shape == cross_base_state[key].shape for key in base_state)
incompatible = cross.load_state_dict(base_state, strict=False)
assert not incompatible.unexpected_keys
assert incompatible.missing_keys
assert all(key.startswith('channel_layers.') for key in incompatible.missing_keys)
cross.channel_layers = nn.ModuleList([nn.Identity() for _ in range(cross.num_layers)])
x = torch.randn(2, 3, 32)
with torch.no_grad():
    assert torch.allclose(base(x), cross(x), atol=1e-6, rtol=1e-5)

torch.manual_seed(16)
mixing_encoder = PatchTFTCrossChannel(**encoder_kwargs).eval()
changed_x = x.clone()
changed_x[:, 0] += 5.0
with torch.no_grad():
    unchanged_channel = mixing_encoder(x)[:, 1]
    mixed_channel = mixing_encoder(changed_x)[:, 1]
assert not torch.allclose(unchanged_channel, mixed_channel)

supervised_kwargs = dict(encoder_kwargs)
supervised_kwargs.pop('pretrain_head')
model = SupervisedPatchTSTCrossChannel(**supervised_kwargs, num_classes=1)
target = torch.tensor([[0.0], [1.0]])
logits = model(x)
assert logits.shape == (2, 1)
nn.BCEWithLogitsLoss()(logits, target).backward()
assert any(p.grad is not None for p in model.encoder.layers.parameters())
assert any(p.grad is not None for p in model.encoder.channel_layers.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())